In [0]:
# ============================================================
# Notebook  : send_price_alerts
# Purpose   : Read Gold table, detect price drops,
#             send Gmail SMTP alerts and log to
#             flight_price_alerts table
# Author    : FlightPriceTracker
# Version   : v1.0
# ============================================================

import uuid
import traceback
import smtplib
from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, TimestampType
)
from delta.tables import DeltaTable

# ============================================================
# SECTION 1 — INIT
# ============================================================

spark = SparkSession.builder \
    .appName("FlightPriceTracker_Alert") \
    .getOrCreate()

# ============================================================
# SECTION 2 — CONSTANTS
# ============================================================

RUN_ID = spark.sql("SELECT run_id FROM workspace.flight_tracker.flight_prices_raw ORDER BY ingested_at DESC LIMIT 1").collect()[0][0]
JOB_NAME           = "FlightPriceTracker"
NOTEBOOK_NAME      = "send_price_alerts"
TASK_SEQUENCE      = 4
TRIGGERED_BY       = "SCHEDULER"
DATABASE           = "workspace.flight_tracker"
AVG_TABLE          = f"{DATABASE}.flight_prices_avg"
CLEAN_TABLE        = f"{DATABASE}.flight_prices_clean"
ALERT_TABLE        = f"{DATABASE}.flight_price_alerts"
AUDIT_TABLE        = f"{DATABASE}.pipeline_audit_log"
ALERT_THRESHOLD    = 15.0
SMTP_HOST          = "smtp.gmail.com"
SMTP_PORT          = 587
SMTP_USER          = "bahadurrakshit4@gmail.com"
SMTP_PASSWORD      = "latr ibao mjdp rvso"
ALERT_RECIPIENT    = "bahadurrakshit4@gmail.com"

def now():
    return datetime.now(timezone.utc)

print(f"RUN_ID     : {RUN_ID}")
print(f"NOTEBOOK   : {NOTEBOOK_NAME}")
print(f"START TIME : {now()}")

# ============================================================
# SECTION 3 — SCHEMAS
# ============================================================

audit_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("job_name",             StringType(),    True),
    StructField("notebook_name",        StringType(),    True),
    StructField("task_sequence",        IntegerType(),   True),
    StructField("cluster_id",           StringType(),    True),
    StructField("start_time",           TimestampType(), True),
    StructField("end_time",             TimestampType(), True),
    StructField("duration_seconds",     IntegerType(),   True),
    StructField("status",               StringType(),    True),
    StructField("records_read",         IntegerType(),   True),
    StructField("records_written",      IntegerType(),   True),
    StructField("records_skipped",      IntegerType(),   True),
    StructField("records_failed",       IntegerType(),   True),
    StructField("error_message",        StringType(),    True),
    StructField("error_stacktrace",     StringType(),    True),
    StructField("retry_attempt",        IntegerType(),   True),
    StructField("triggered_by",         StringType(),    True),
    StructField("databricks_job_run_id",StringType(),    True),
    StructField("created_at",           TimestampType(), True),
])

alert_schema = StructType([
    StructField("alert_id",            StringType(),    True),
    StructField("run_id",              StringType(),    True),
    StructField("route_id",            StringType(),    True),
    StructField("airline",             StringType(),    True),
    StructField("airline_code",        StringType(),    True),
    StructField("cabin_class",         StringType(),    True),
    StructField("current_price",       DoubleType(),    True),
    StructField("avg_7day_price",      DoubleType(),    True),
    StructField("min_7day_price",      DoubleType(),    True),
    StructField("drop_pct",            DoubleType(),    True),
    StructField("price_trend",         StringType(),    True),
    StructField("departure_datetime",  TimestampType(), True),
    StructField("seats_available",     IntegerType(),   True),
    StructField("alert_type",          StringType(),    True),
    StructField("alert_severity",      StringType(),    True),
    StructField("email_recipient",     StringType(),    True),
    StructField("email_subject",       StringType(),    True),
    StructField("email_status",        StringType(),    True),
    StructField("email_error_message", StringType(),    True),
    StructField("alert_sent_at",       TimestampType(), True),
    StructField("alert_created_at",    TimestampType(), True),
])

# ============================================================
# SECTION 4 — AUDIT HELPER
# ============================================================

def log_audit(
    run_id, job_name, notebook_name, task_sequence,
    start_time, end_time, status,
    records_read=0, records_written=0,
    records_skipped=0, records_failed=0,
    error_message=None, error_stacktrace=None,
    retry_attempt=0, triggered_by="SCHEDULER"
):
    duration = int((end_time - start_time).total_seconds())
    audit_data = [{
        "run_id"               : run_id,
        "job_name"             : job_name,
        "notebook_name"        : notebook_name,
        "task_sequence"        : task_sequence,
        "cluster_id"           : "serverless",
        "start_time"           : start_time,
        "end_time"             : end_time,
        "duration_seconds"     : duration,
        "status"               : status,
        "records_read"         : records_read,
        "records_written"      : records_written,
        "records_skipped"      : records_skipped,
        "records_failed"       : records_failed,
        "error_message"        : error_message if error_message else "",
        "error_stacktrace"     : error_stacktrace if error_stacktrace else "",
        "retry_attempt"        : retry_attempt,
        "triggered_by"         : triggered_by,
        "databricks_job_run_id": "",
        "created_at"           : now()
    }]
    audit_df = spark.createDataFrame(audit_data, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    print(f"[AUDIT] {status} logged for {notebook_name}")

# ============================================================
# SECTION 5 — ALERT SEVERITY
# ============================================================

def get_severity(drop_pct):
    if drop_pct >= 40:
        return "HIGH"
    elif drop_pct >= 25:
        return "MEDIUM"
    else:
        return "LOW"

# ============================================================
# SECTION 6 — EMAIL BUILDER
# ============================================================

def build_email(row):
    severity = get_severity(row["drop_pct"])
    subject  = f"[{severity}] Flight Price Drop — {row['route_id']} | {row['airline']} | {row['cabin_class']}"
    body     = f"""
    <html>
    <body style="font-family: Arial, sans-serif; padding: 20px;">
        <h2 style="color: {'#d32f2f' if severity == 'HIGH' else '#f57c00' if severity == 'MEDIUM' else '#388e3c'};">
            Flight Price Drop Alert — {severity} Priority
        </h2>
        <table style="border-collapse: collapse; width: 100%;">
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Route</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">{row['route_id']}</td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Airline</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">{row['airline']} ({row['airline_code']})</td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Cabin Class</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">{row['cabin_class']}</td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Current Price</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; color: green;"><b>${row['latest_price']:.2f}</b></td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>7-Day Average</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">${row['avg_7day_price']:.2f}</td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Price Drop</b></td>
                <td style="padding: 8px; border: 1px solid #ddd; color: red;"><b>{row['drop_pct']:.2f}%</b></td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Price Trend</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">{row['price_trend']}</td></tr>
            <tr><td style="padding: 8px; border: 1px solid #ddd;"><b>Alert Generated At</b></td>
                <td style="padding: 8px; border: 1px solid #ddd;">{now()} UTC</td></tr>
        </table>
        <p style="color: #888; font-size: 12px; margin-top: 20px;">
            FlightPriceTracker pipeline — Run ID: {RUN_ID}
        </p>
    </body>
    </html>
    """
    return subject, body

# ============================================================
# SECTION 7 — SEND EMAIL
# ============================================================

def send_email(subject, body):
    try:
        msg            = MIMEMultipart("alternative")
        msg["Subject"] = subject
        msg["From"]    = SMTP_USER
        msg["To"]      = ALERT_RECIPIENT
        msg.attach(MIMEText(body, "html"))

        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.starttls()
            server.login(SMTP_USER, SMTP_PASSWORD)
            server.sendmail(SMTP_USER, ALERT_RECIPIENT, msg.as_string())

        print(f"[EMAIL] Sent: {subject}")
        return "SUCCESS", ""

    except Exception as e:
        print(f"[EMAIL] Failed: {str(e)}")
        return "FAILED", str(e)

# ============================================================
# SECTION 8 — LOG ALERT
# ============================================================

def log_alert(row, subject, email_status, email_error):
    alert_data = [{
        "alert_id"            : str(uuid.uuid4()),
        "run_id"              : RUN_ID,
        "route_id"            : row["route_id"],
        "airline"             : row["airline"],
        "airline_code"        : row["airline_code"],
        "cabin_class"         : row["cabin_class"],
        "current_price"       : float(row["latest_price"]),
        "avg_7day_price"      : float(row["avg_7day_price"]),
        "min_7day_price"      : float(row["min_7day_price"]),
        "drop_pct"            : float(row["drop_pct"]),
        "price_trend"         : row["price_trend"],
        "departure_datetime"  : None,
        "seats_available"     : 0,
        "alert_type"          : "PRICE_DROP",
        "alert_severity"      : get_severity(row["drop_pct"]),
        "email_recipient"     : ALERT_RECIPIENT,
        "email_subject"       : subject,
        "email_status"        : email_status,
        "email_error_message" : email_error,
        "alert_sent_at"       : now() if email_status == "SUCCESS" else None,
        "alert_created_at"    : now()
    }]
    alert_df = spark.createDataFrame(alert_data, schema=alert_schema)
    alert_df.write.format("delta").mode("append").saveAsTable(ALERT_TABLE)
    print(f"[ALERT LOG] Logged for {row['route_id']} — {row['airline']}")

# ============================================================
# SECTION 9 — MAIN EXECUTION
# ============================================================

start_time      = now()
records_read    = 0
records_written = 0
records_skipped = 0
records_failed  = 0
pipeline_status = "SUCCESS"
pipeline_error  = None
pipeline_trace  = None

try:
    print(f"\n[READ] Reading current Gold records from {AVG_TABLE}...")

    gold_df = spark.sql(f"""
        SELECT
            route_id,
            airline,
            airline_code,
            cabin_class,
            avg_7day_price,
            min_7day_price,
            latest_price,
            price_drop_pct AS drop_pct,
            price_trend
        FROM {AVG_TABLE}
        WHERE is_current      = true
        AND   price_drop_pct >= {ALERT_THRESHOLD}
    """)

    records_read = gold_df.count()
    print(f"[READ] Price drop records found: {records_read}")

    if records_read == 0:
        print("[READ] No price drops detected above threshold.")
        pipeline_status = "SUCCESS"

    else:
        alert_rows = gold_df.collect()

        for row in alert_rows:
            subject, body           = build_email(row)
            email_status, email_err = send_email(subject, body)
            log_alert(row, subject, email_status, email_err)

            if email_status == "SUCCESS":
                records_written += 1
            else:
                records_failed  += 1

        print(f"[SUMMARY] Alerts Sent: {records_written} | Failed: {records_failed}")

except Exception as e:
    pipeline_status = "FAILED"
    pipeline_error  = str(e)
    pipeline_trace  = traceback.format_exc()
    print(f"[ERROR] {pipeline_error}")

finally:
    end_time = now()
    log_audit(
        run_id           = RUN_ID,
        job_name         = JOB_NAME,
        notebook_name    = NOTEBOOK_NAME,
        task_sequence    = TASK_SEQUENCE,
        start_time       = start_time,
        end_time         = end_time,
        status           = pipeline_status,
        records_read     = records_read,
        records_written  = records_written,
        records_skipped  = records_skipped,
        records_failed   = records_failed,
        error_message    = pipeline_error,
        error_stacktrace = pipeline_trace,
        triggered_by     = TRIGGERED_BY
    )
    print(f"\n[DONE] {NOTEBOOK_NAME} — Status: {pipeline_status}")
    print(f"[DONE] Read: {records_read} | Sent: {records_written} | Failed: {records_failed}")
    dbutils.notebook.exit(RUN_ID)